# AFML SLM — Kaggle trainingPretraining for the GQA/DPO project. Run the cells in order.**Before you start:** Settings (right panel) → Accelerator → **GPU T4 x2** (or P100),and Internet → **On**.**Session limits.** A GPU session is killed at 12 hours and the weekly quota isabout 30 GPU-hours. Checkpoints are written every `checkpoint_interval` steps,so a killed run resumes from the last one — this is verified by`scripts/test_resume.py`, which passes bit-exact.**Across sessions:** click *Save Version* → *Save & Run All* when a run finishes.That persists `/kaggle/working`. To continue later, attach this notebook's outputas an input dataset and set `PREV_OUTPUT` in the resume cell.

## 1. Code

In [ ]:
import os, subprocess, sys, shutilfrom pathlib import PathREPO   = "https://github.com/Shikshith05/AFML_miniproject.git"BRANCH = "main"ROOT   = Path("/kaggle/working/AFML_miniproject")if ROOT.exists():    shutil.rmtree(ROOT)subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(ROOT)], check=True)os.chdir(ROOT)sys.path.insert(0, str(ROOT / "src"))print("at", Path.cwd())print(subprocess.run(["git","log","--oneline","-1"], capture_output=True, text=True).stdout)

In [ ]:
# torch and numpy are preinstalled on Kaggle; only these are missing!pip install -q tokenizers datasets ftfy

## 2. GPU check

In [ ]:
import torchprint("torch           ", torch.__version__)print("cuda available  ", torch.cuda.is_available())if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print("device          ", p.name)    print("memory          ", f"{p.total_memory/1e9:.1f} GB")    print("bf16 supported  ", torch.cuda.is_bf16_supported())    print()    print("-> use precision 'bf16' if supported, else 'fp16'")else:    raise SystemExit("No GPU. Settings -> Accelerator -> GPU, then rerun.")

## 3. DataBuilds `data/train.bin` and `data/val.bin` from TinyStories using the tokenizercommitted in the repo.If you attached a previous output as an input dataset, set `DATA_FROM` to itspath to skip the rebuild — it takes about 40 minutes of session time.

In [ ]:
from pathlib import Pathimport shutilDATA_FROM = None          # e.g. "/kaggle/input/afml-slm-data/data"LIMIT     = None          # None = full dataset; set e.g. 200000 to go fasterdata_dir = Path("data")if DATA_FROM and Path(DATA_FROM).exists():    data_dir.mkdir(exist_ok=True)    for f in Path(DATA_FROM).glob("*"):        shutil.copy(f, data_dir / f.name)    print("copied prepared data from", DATA_FROM)elif (data_dir / "train.bin").exists():    print("data already present")else:    cmd = "python scripts/prepare_data.py" + (f" --limit {LIMIT}" if LIMIT else "")    print(cmd)    get_ipython().system(cmd)for f in sorted(data_dir.glob("*.bin")):    print(f"{f.name:>12}  {f.stat().st_size/1e6:>8.1f} MB")

## 4. ProbeFinds the largest batch size that fits, measures throughput, and converts awall-clock budget into `max_steps`.Run this **once per GPU type**. Use the same numbers for both the MHA and theGQA run — matched conditions are what make the ablation valid.

In [ ]:
!python scripts/probe_batch_size.py --preset gqa --budget-hours 3

## 5. TrainFill in `MICRO_BS`, `ACCUM` and `MAX_STEPS` from the probe output above.Run MHA first, then GQA, changing only `PRESET`. Everything else must stayidentical, seed included.

In [ ]:
PRESET    = "mha"      # "mha" first, then "gqa"MICRO_BS  = 16         # <- from the probeACCUM     = 8          # <- from the probeMAX_STEPS = 20000      # <- from the probePRECISION = "fp16"     # "bf16" if cell 2 said it is supportedimport syssys.argv = [    "pretrain.py",    "--preset", PRESET,    "--run-name", f"{PRESET}_base",    "--micro-batch-size", str(MICRO_BS),    "--max-steps", str(MAX_STEPS),    "--precision", PRECISION,]from slm.model.config import gqa_base, mha_baselinecfg = {"mha": mha_baseline, "gqa": gqa_base}[PRESET]()cfg.run_name            = f"{PRESET}_base"cfg.train.micro_batch_size = MICRO_BScfg.train.grad_accum_steps = ACCUMcfg.train.max_steps        = MAX_STEPScfg.train.precision        = PRECISIONprint(cfg.summary())

In [ ]:
from slm.model.transformer import Transformerfrom slm.data.tinystories import make_batch_providerfrom slm.training.pretrain import pretrainmodel     = Transformer(cfg.model)get_batch = make_batch_provider(cfg)# resume=True picks up the newest checkpoint in checkpoints/<run_name>/pretrain(model, cfg, get_batch, resume=True)

## 6. Resuming after a killed session1. *Save Version* → *Save & Run All* so `/kaggle/working` is kept2. In the new session: *Add Input* → *Notebook Output* → pick this notebook3. Run the cell below before cell 5, then run cell 5 as usual — `resume=True`   finds the checkpoint and continues

In [ ]:
# Point this at the attached previous output, then rerun the training cells.PREV_OUTPUT = None    # e.g. "/kaggle/input/afml-slm-mha/AFML_miniproject/checkpoints"from pathlib import Pathimport shutilif PREV_OUTPUT and Path(PREV_OUTPUT).exists():    dest = Path("checkpoints")    dest.mkdir(exist_ok=True)    for run in Path(PREV_OUTPUT).iterdir():        if run.is_dir():            shutil.copytree(run, dest / run.name, dirs_exist_ok=True)            ckpts = sorted(p.name for p in (dest / run.name).glob("*.pt"))            print(run.name, "->", ckpts[-3:] if ckpts else "none")else:    print("PREV_OUTPUT not set -- starting fresh")

## 7. Results`results/<run_name>/metrics.jsonl` holds every logged step. Download it from theOutput tab and commit it to the repo — those numbers go into the paper.

In [ ]:
from slm.training.utils import read_metricsfrom pathlib import Pathfor m in sorted(Path("results").glob("*/metrics.jsonl")):    rows  = list(read_metrics(m))    evals = [r for r in rows if r.get("event") == "eval"]    print(f"{m.parent.name:>12}  {len(rows):>5} rows", end="")    if evals:        print(f"   last val_loss {evals[-1]['val_loss']:.4f}"              f"   ppl {evals[-1]['val_ppl']:.2f}")    else:        print()